# 04 — Combined Clinical + Global PCA Clustering

Clusters OASIS-1 patients using **all available information at once**:
the 10 clinical variables and the 50 global PCA components from HOG features.
A Generalized Gower distance matrix handles the mixed variable types
(continuous, ordinal, binary) across all 58 retained columns together.

**Input :** `oasis_combined.csv` — already contains both clinical columns and PC1–PC50  
**Output:** `combined_clinical_pca_global_results.csv` — `patient_id`, `cluster`, `CDR`

In [1]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "robust-mixed-dist", "kmedoids", "scikit-learn", "-q"],
    check=True
)

CompletedProcess(args=['C:\\Users\\aregk\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', 'robust-mixed-dist', 'kmedoids', 'scikit-learn', '-q'], returncode=0)

In [2]:
import os
import numpy as np
import pandas as pd

from robust_mixed_dist.mixed import generalized_gower_dist_matrix
import kmedoids

## Step 1 — Load data

`oasis_combined.csv` already contains both the HOG global PCA components
(PC1–PC50) and the clinical variables in a single file.  CDR is separated
immediately and never used during clustering — only for post-hoc evaluation.

In [3]:
DATA_DIR   = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
CSV_PATH   = os.path.join(DATA_DIR, "method_1_HOG_PCA", "oasis_combined.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "notebooks", "combined_clinical_pca_global_results.csv")

CLINICAL_COLS = ["Age", "M/F", "Hand", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF", "Delay"]
PC_COLS       = [f"PC{i}" for i in range(1, 51)]

raw = pd.read_csv(CSV_PATH)
df  = raw[["patient_id"] + CLINICAL_COLS + PC_COLS].copy()
cdr = raw[["patient_id", "CDR"]].copy()   # evaluation only

print(f"Loaded : {df.shape[0]} patients, {df.shape[1]-1} feature columns")
print(f"Clinical columns : {len(CLINICAL_COLS)}")
print(f"PC columns       : {len(PC_COLS)}")
df.head(3)

Loaded : 416 patients, 60 feature columns
Clinical columns : 10
PC columns       : 50


,patient_id,Age,M/F,Hand,Educ,SES,MMSE,eTIV,nWBV,ASF,...,PC41,PC42,PC43,PC44,PC45,PC46,PC47,PC48,PC49,PC50
0,OAS1_0001,74,F,R,2.0,3.0,29.0,1344,0.743,1.306,...,0.184081,-0.137001,0.414360,-0.044129,0.150283,0.104149,0.000962,0.046894,-0.009280,-0.035229
1,OAS1_0002,55,F,R,4.0,1.0,29.0,1147,0.810,1.531,...,-0.044508,0.141253,0.120110,0.222860,0.283561,0.065880,0.102409,0.252023,-0.029055,-0.200865
2,OAS1_0003,73,F,R,4.0,3.0,27.0,1454,0.708,1.207,...,0.191160,-0.026945,-0.047394,0.141422,-0.150229,-0.046390,0.051265,0.008706,0.092923,0.167697


## Step 2 — Missing value analysis

The same issues as in notebook 02 apply to the clinical columns:

| Column | Situation | Action |
|--------|-----------|--------|
| `Delay` | 100 % missing | **Drop** |
| `Hand` | Constant (`R`) | **Drop** — zero variance |
| `Educ`, `SES`, `MMSE` | ~43–48 % missing | **Impute** with median |
| All PC columns | 0 % missing | Keep as-is |

In [4]:
missing    = df.isnull().sum().rename("missing")
pct        = (missing / len(df) * 100).round(1).rename("%")
unique_cnt = df.nunique().rename("unique_vals")

# Show only columns that have missing values or are the clinical columns
show_cols  = [c for c in df.columns if missing[c] > 0 or c in CLINICAL_COLS + ["patient_id"]]
print(pd.concat([missing, pct, unique_cnt], axis=1).loc[show_cols].to_string())
print(f"\nPC1–PC50 missing  : {missing[PC_COLS].sum()} total")

            missing      %  unique_vals
patient_id        0    0.0          416
Age               0    0.0           73
M/F               0    0.0            2
Hand              0    0.0            1
Educ            181   43.5            5
SES             200   48.1            5
MMSE            181   43.5           17
eTIV              0    0.0          301
nWBV              0    0.0          182
ASF               0    0.0          275
Delay           416  100.0            0

PC1–PC50 missing  : 0 total


In [5]:
df = df.drop(columns=["Delay", "Hand"])
print("Dropped: Delay (100% missing), Hand (constant 'R')")

for col in ["Educ", "SES", "MMSE"]:
    med = df[col].median()
    n_filled = df[col].isnull().sum()
    df[col] = df[col].fillna(med)
    print(f"  {col}: filled {n_filled} NaNs with median = {med}")

print(f"\nRemaining missing : {df.isnull().sum().sum()}")
print(f"Final feature set : {df.shape[1]-1} columns  "
      f"(7 clinical quantitative + 1 binary + 50 PC)")

Dropped: Delay (100% missing), Hand (constant 'R')
  Educ: filled 181 NaNs with median = 3.0
  SES: filled 200 NaNs with median = 2.0
  MMSE: filled 181 NaNs with median = 29.0

Remaining missing : 0
Final feature set : 58 columns  (7 clinical quantitative + 1 binary + 50 PC)


## Step 3 — Generalized Gower distance matrix

All 58 features are passed together.  The function requires columns ordered
as **quantitative → binary → multi-class**:

| Type | Columns | Count |
|------|---------|-------|
| Quantitative (`p1`) | Age, Educ, SES, MMSE, eTIV, nWBV, ASF, PC1–PC50 | 57 |
| Binary (`p2`) | M/F (0=F, 1=M) | 1 |
| Multi-class (`p3`) | *(none)* | 0 |

Result is a symmetric 416×416 matrix with values in [0, 1].

In [6]:
# Encode M/F as binary integer
df["MF_bin"] = df["M/F"].map({"F": 0, "M": 1}).astype(int)

quant_cols  = ["Age", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"] + PC_COLS
binary_cols = ["MF_bin"]

p1 = len(quant_cols)    # 57
p2 = len(binary_cols)   # 1
p3 = 0

X = df[quant_cols + binary_cols].values.astype(float)

print(f"Feature matrix shape : {X.shape}")
print(f"  p1 (quantitative)  : {p1}  (7 clinical + 50 PC)")
print(f"  p2 (binary)        : {p2}  (M/F)")
print(f"  p3 (multi-class)   : {p3}")

D = generalized_gower_dist_matrix(
    X,
    p1=p1, p2=p2, p3=p3,
    d1="minkowski",
    d2="sokal",
    d3="hamming",
    q=1
)

print(f"\nDistance matrix shape : {D.shape}")
print(f"Value range           : [{D.min():.4f}, {D.max():.4f}]")
print(f"Is symmetric          : {np.allclose(D, D.T)}")

Feature matrix shape : (416, 58)
  p1 (quantitative)  : 57  (7 clinical + 50 PC)
  p2 (binary)        : 1  (M/F)
  p3 (multi-class)   : 0

Distance matrix shape : (416, 416)
Value range           : [0.0000, 5.4465]
Is symmetric          : True


## Step 4 — K-medoids clustering (FasterPAM, k=4)

In [7]:
K = 4

result = kmedoids.fasterpam(D, medoids=K, random_state=42)
labels = np.array(result.labels)

print(f"Loss (sum of distances to medoids): {result.loss:.4f}")
print(f"Medoid patient IDs : {df['patient_id'].iloc[list(result.medoids)].tolist()}")
print()
sizes = pd.Series(labels).value_counts().sort_index().rename("count")
print("Cluster sizes:")
print(sizes.to_string())

Loss (sum of distances to medoids): 221.2981
Medoid patient IDs : ['OAS1_0386', 'OAS1_0070', 'OAS1_0389', 'OAS1_0101']

Cluster sizes:
0    122
1    134
2     76
3     84


## Step 5 — Evaluation: cluster vs CDR

CDR was never used during clustering.  Merge it back in and cross-tabulate
against cluster assignments.  Patients without CDR are excluded from the table.

In [8]:
results = df[["patient_id"]].copy()
results["cluster"] = labels
results = results.merge(cdr, on="patient_id", how="left")

print(f"Patients with CDR    : {results['CDR'].notnull().sum()}")
print(f"Patients without CDR : {results['CDR'].isnull().sum()}")
print()

has_cdr = results[results["CDR"].notnull()].copy()
has_cdr["CDR"] = has_cdr["CDR"].astype(str)

ct = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    margins=True,
    margins_name="Total"
)
print("Cluster × CDR (counts):")
ct

Patients with CDR    : 235
Patients without CDR : 181

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,32,17,11,0,60
1,65,22,8,1,96
2,21,14,5,0,40
3,17,17,4,1,39
Total,135,70,28,2,235


In [9]:
ct_norm = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    normalize="index"
).round(3)

print("Row-normalised (CDR proportion within each cluster):")
ct_norm

Row-normalised (CDR proportion within each cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.533,0.283,0.183,0.000
1,0.677,0.229,0.083,0.010
2,0.525,0.350,0.125,0.000
3,0.436,0.436,0.103,0.026


## Step 6 — Save results

In [10]:
results.to_csv(OUTPUT_CSV, index=False)

print(f"Saved : {OUTPUT_CSV}")
print(f"Shape : {results.shape}  (rows=patients, cols=patient_id, cluster, CDR)")
print()
print(results.head(5).to_string(index=False))

Saved : C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\combined_clinical_pca_global_results.csv
Shape : (416, 3)  (rows=patients, cols=patient_id, cluster, CDR)

patient_id  cluster  CDR
 OAS1_0001        1  0.0
 OAS1_0002        1  0.0
 OAS1_0003        0  0.5
 OAS1_0004        3  NaN
 OAS1_0005        2  NaN
